(sec:T3:SF:interpretacion)=
# Interpretación espectral de la serie de Fourier

Desde un punto de vista espectral, la serie de Fourier proporciona una representación discreta del contenido en frecuencia de una señal periódica. Cada coeficiente $c_k$ está asociado a una componente armónica de frecuencia $k\omega_0$, de modo que la periodicidad en el dominio temporal se traduce en una discretización del espectro en el dominio de la frecuencia. Como ejemplo, se muestra en la figura la serie de Fourier correspondiente a un tren de pulsos triangulares.

```{figure} figures/T3/signal_tri_periodic2.svg
:name: figs:T3:SF:interpretacion_tiempo
:width: 80%
:alt: Tren de pulsos triangulares.
:align: center
```
```{figure} figures/T3/signal_tri_periodic2_dk.svg
:name: figs:T3:SF:interpretacion_frecuencia
:width: 80%
:alt: Serie de Fourier correspondiente a un tren de pulsos triangulares.
:align: center
```

Esta interpretación permite analizar de forma cualitativa el comportamiento de las señales y los sistemas:
- Señales cuyos coeficientes son significativos únicamente para valores pequeños de $|k|$ presentan una variación lenta en el tiempo.
- La presencia de armónicos de orden elevado se asocia a variaciones rápidas o a la existencia de discontinuidades.

:::{tip} Interpretación espectral
En la siguiente gráfica interactiva se analiza la relación entre la forma de onda de la señal $\tilde{x}(t)$ y el módulo de los coeficientes de su serie de Fourier, $|c_k|$, siendo:
```{math}
    \tilde{x}(t) \propto \tanh(s \cdot (\sin(\omega_0 t) + a))
```
Se pueden variar la Abrupticidad ($s$) y el Offset ($a$).

Observa:
- **Suavidad vs. Ancho de banda**: Cuando la señal es suave ($s$ bajo), varía lentamente y su energía se concentra en frecuencias bajas. Al aumentar la abrupticidad ($s$ alto), las transiciones se vuelven rápidas, lo que obliga a que aparezcan armónicos de orden elevado para construir esos cambios bruscos.
- **Potencia media**: Al hacer la señal más cuadrada (mayor $s$), los coeficientes fundamentales crecen. Esto ocurre porque la señal "se ensancha" en el tiempo, aumentando su área y potencia media[^foot1].
- **Simetría y armónicos pares**: Si el offset es nulo ($a=0$), la señal tiene *simetría de media onda*[^foot2] y sólo contiene armónicos impares. Al añadir asimetría ($a \neq 0$), se rompe esta condición y aparecen los armónicos pares ($k=\pm 2, \pm 4 \dots$).
- **Componente continua** ($c_0$): Este coeficiente representa el valor medio de la señal. Con $a=0$, la señal es simétrica respecto al eje horizontal y su media es nula. Al aumentar el offset, el valor medio deja de ser cero, haciendo que $c_0\neq0$.
:::

[^foot1]: Ver {ref}`subsec:T3:SF:parseval`.
[^foot2]: La **simetría de media onda** ocurre cuando la parte negativa de la señal es un reflejo invertido de la parte positiva, desplazada medio período ($x(t + T_0/2) = -x(t)$). En el dominio espectral, esto causa la cancelación de todos los armónicos pares ($c_2, c_4 \dots$) y la componente continua ($c_0$).

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))

from utils.plot_helpers import style_math_axes

In [16]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Div
from bokeh.io import output_notebook

# Importamos las funciones auxiliares
from utils.plot_helpers import style_math_axes, add_math_ticks

output_notebook()

# ==========================================
# 1. PARÁMETROS INICIALES
# ==========================================
VAL_STEEPNESS = 0.5  
VAL_OFFSET = 0.0     # Offset inicial (0 = simétrico)
N_POINTS = 400       
N_HARMONICS = 20     

COLOR_TIME = "blue" 
COLOR_FREQ = "blue" 

t_min, t_max = -2.5, 2.5
t_vals = np.linspace(t_min, t_max, N_POINTS)

# ==========================================
# 2. GENERACIÓN DE DATOS
# ==========================================

# A. Datos Tiempo
# x(t) = tanh( s * (sin(pi*t) + offset) )
def generate_signal(t_arr, s, off):
    if s < 1e-3: s = 1e-3
    # Normalizamos respecto al pico máximo posible para mantener la escala visual
    norm = np.tanh(s * (1.0 + abs(off)))
    return np.tanh(s * (np.sin(np.pi * t_arr) + off)) / norm

y_vals = generate_signal(t_vals, VAL_STEEPNESS, VAL_OFFSET)
source_time = ColumnDataSource(data=dict(t=t_vals, y=y_vals))

# B. Datos Frecuencia
k_vals = np.arange(-N_HARMONICS, N_HARMONICS + 1)
t_period = np.linspace(-1, 1, N_POINTS, endpoint=False) 

def calc_dft(s, off):
    if s < 1e-3: s = 1e-3
    norm = np.tanh(s * (1.0 + abs(off)))
    y_p = np.tanh(s * (np.sin(np.pi * t_period) + off)) / norm
    
    mags = []
    for k in k_vals:
        angle = k * np.pi * t_period
        re = np.mean(y_p * np.cos(angle))
        im = np.mean(y_p * (-np.sin(angle)))
        
        mag = np.sqrt(re**2 + im**2)
        if mag < 1e-4: mag = 0
        mags.append(mag)
        
    return np.array(mags)

mag_vals = calc_dft(VAL_STEEPNESS, VAL_OFFSET)
source_freq = ColumnDataSource(data=dict(k=k_vals, mag=mag_vals, zeros=np.zeros_like(k_vals)))


# ==========================================
# 3. GRÁFICOS
# ==========================================

# --- TIEMPO ---
p_time = figure(width=600, height=300) 
style_math_axes(p_time, x_range=(t_min, t_max), y_range=(-1.3, 1.3), xlabel="t", ylabel=r"$$\tilde{x}(t)$$")
add_math_ticks(p_time, yticks=[-1, 1], ytick_labels=["-1", "1"], tick_len=5)

p_time.line('t', 'y', source=source_time, color=COLOR_TIME, line_width=3)

# --- FRECUENCIA ---
p_freq = figure(width=600, height=300)
style_math_axes(p_freq, x_range=(-N_HARMONICS-1, N_HARMONICS+1), y_range=(0, 0.9), xlabel="k", ylabel=r"$$|c_k|$$")

p_freq.segment(x0='k', y0='zeros', x1='k', y1='mag', source=source_freq, color=COLOR_FREQ, line_width=3)
p_freq.scatter('k', 'mag', source=source_freq, color=COLOR_FREQ, size=8, marker="circle")


# ==========================================
# 4. INTERACTIVIDAD
# ==========================================
s_steep = Slider(start=0.1, end=10.0, value=VAL_STEEPNESS, step=0.1, title=r"Abrupticidad ($$s$$)")
s_offset = Slider(start=0.0, end=1.0, value=VAL_OFFSET, step=0.05, title=r"Offset (Asimetría)")

callback = CustomJS(
    args=dict(source_t=source_time, source_f=source_freq, s_steep=s_steep, s_offset=s_offset),
    code="""
    const s = s_steep.value;
    const off = s_offset.value;
    const PI = Math.PI;
    
    // Normalización dinámica para que la señal no se salga de gráfica
    let norm = Math.tanh(s * (1.0 + Math.abs(off)));
    if (Math.abs(norm) < 1e-9) norm = 1.0;

    // --- 1. TIEMPO ---
    const t = source_t.data['t'];
    const y = source_t.data['y'];

    for (let i = 0; i < t.length; i++) {
        y[i] = Math.tanh(s * (Math.sin(PI * t[i]) + off)) / norm;
    }
    source_t.change.emit();

    // --- 2. FRECUENCIA (DFT) ---
    const N_integ = 200; 
    const k_arr = source_f.data['k'];
    const mag = source_f.data['mag'];
    
    let y_p = new Float32Array(N_integ);
    let t_p = new Float32Array(N_integ);
    
    // Generamos un periodo auxiliar
    for(let i=0; i<N_integ; i++){
        let ti = -1 + (2 * i / N_integ);
        t_p[i] = ti;
        y_p[i] = Math.tanh(s * (Math.sin(PI * ti) + off)) / norm;
    }
    
    for (let i = 0; i < k_arr.length; i++) {
        let k = k_arr[i];
        let sum_re = 0.0;
        let sum_im = 0.0;
        
        for (let j = 0; j < N_integ; j++) {
            let angle = PI * k * t_p[j]; 
            sum_re += y_p[j] * Math.cos(angle);
            sum_im += y_p[j] * (-Math.sin(angle));
        }
        
        let re = sum_re / N_integ;
        let im = sum_im / N_integ;
        let m = Math.sqrt(re*re + im*im);
        
        if (m < 1e-3) m = 0;
        mag[i] = m;
    }
    source_f.change.emit();
""")

s_steep.js_on_change('value', callback)
s_offset.js_on_change('value', callback)

# ==========================================
# 5. LAYOUT
# ==========================================

caption_text = """
<div style="font-family: sans-serif; margin-top: 15px; font-size: 14px; color: #444;">
    <p><b>Interpretación espectral:</b> suavidad/abrupticidad y simetría de la señal.</p>
</div>
"""
caption = Div(text=caption_text)

# Organizamos los sliders en una fila
layout = column(row(s_steep, s_offset), p_time, p_freq, caption, sizing_mode="scale_width")
show(layout)

Loading BokehJS ...

Por otro lado, teniendo en cuenta que las exponenciales complejas son autofunciones de los sistemas LTI, el desarrollo en serie de Fourier a la salida de un sistema LTI:

```{figure} figures/T3/diag4.svg
:name: figs:T3:SF:LTI_SF_salida
:width: 80%
:alt: Serie de Fourier correspondiente a un tren de pulsos triangulares.
:align: center
```

Tendrá como coeficientes:
```{math}
	\tilde{y}(t) \SF d_k = H(k\omega_0)\,c_k
```
donde $H(k\omega_0)$ es la respuesta en frecuencia del sistema a las frecuencias armónicas de la fundamental.

Asimismo, esta representación discreta del espectro constituye el punto de partida para establecer relaciones estructurales con otros conceptos del análisis de Fourier, como la transformada de Fourier de señales aperiódicas y la interpretación del muestreo como fenómeno dual de la periodicidad.